# 88 Ablations report (CPU runtime)

1. **The base model on validation**: its greedy answers (already made for the pool, same model, prompt and settings as the zero-shot dev run) are scored like any model. Needed for the first row of the component ladder.
2. **The ablation tables** (`python -m effsql.ablate report`): the component ladder, the verifier ablations, the sample-count curve, the prompt sources, results by difficulty, what the verifier learned, the 1 s cap, the cost spread of the model's own answers and worked examples. Saved as `ablations_report/report.txt` and `report.json` (the data for the figures).

Every choice is recomputed from saved files: the candidates, the verifiers' scores and the answer-time results. A chosen candidate's correctness is its label, which equals the evaluation's EX for every final answer. Nothing is chosen on dev or mini-dev.

**Runtime > Change runtime type > CPU**, then *Run all*. About 15 minutes; after a disconnect press *Run all* again.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import subprocess


def run(cmd):
    """Run a command, show its output, and stop "Run all" if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"failed (exit {p.returncode}): {cmd}")


import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
for j in ("judge_untrained", "judge_v2_own", "judge_v2_nocost", "judge_v2_base"):
    for s in ("val", "dev", "minidev"):
        assert os.path.exists(f"{R}/judge_cand_v1/scores_{j}_{s}.done.json"), f"run 87_ablations first ({j}, {s})"
assert os.path.exists(f"{R}/judge_cand_sft/labels_minidev.jsonl"), "run 87_ablations first (B3 labels)"
VAL_DBS = ",".join(json.load(open(f"{R}/sft_v1/export_meta.json"))["holdout_db_ids"])

### 1. The base model (zero-shot) on validation

In [ ]:
run(f"effsql stage --split train --db-ids {VAL_DBS}")
if not os.path.exists(f"{R}/zero_shot_v1/predictions_val.jsonl"):
    run("python -m effsql.ablate zero-shot-val --run zero_shot_v1")
run("python -m effsql.train eval-val --run zero_shot_v1")

### 2. The ablation tables

In [ ]:
run("python -m effsql.ablate report --out ablations_report")

### Done: disconnect the runtime